In [ ]:
!pip install kaggle==1.5.16

In [ ]:
! chmod 600 .kaggle/kaggle.json

In [ ]:
! kaggle competitions download Walmart-Recruiting-Store-Sales-Forecasting

In [ ]:
! unzip Walmart-Recruiting-Store-Sales-Forecasting.zip

In [ ]:
! unzip features.csv.zip

In [ ]:
! unzip test.csv.zip

In [ ]:
! unzip train.csv.zip

# N-BEATS — Walmart Store Sales Forecasting

In [ ]:
! pip install optuna

In [ ]:
! pip install mlflow

In [ ]:
!pip install "numpy<2" "torchvision==0.17.0" "torch==2.2.0" "neuralforecast==1.7.4" optuna mlflow dagshub wandb -q --force-reinstall

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, os
import wandb
import mlflow
import mlflow.pyfunc
import optuna

from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from neuralforecast.auto import AutoNBEATS
from neuralforecast.losses.pytorch import MAE

optuna.logging.set_verbosity(optuna.logging.WARNING)

WANDB_ENTITY  = 'ikvas22-free-university-of-tbilisi'
WANDB_PROJECT = 'Walmart Weekly Sales Prediction'
WANDB_GROUP   = 'NBEATS'

MLFLOW_EXPERIMENT = 'NBEATS_Training'
MLFLOW_MODEL_NAME = 'nbeats_walmart_best'

TRAIN_PATH    = 'train.csv'
FEATURES_PATH = 'features.csv'
STORES_PATH   = 'stores.csv'

H          = 4    # forecast horAizon (weeks)
INPUT_SIZE = 52   # lookback window (1 year)
N_TRIALS   = 30   # Optuna trials
VAL_START  = '2012-04-01'

wandb.login()
# mlflow.set_experiment(MLFLOW_EXPERIMENT)

print('Setup OK')

## 1. Pre-processing

In [21]:
run = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'preprocessing',
    name     = 'NBEATS_Preprocessing',
)

# ── Load & merge ───────────────────────────────────────────────
train_raw    = pd.read_csv(TRAIN_PATH,    parse_dates=['Date'])
features_raw = pd.read_csv(FEATURES_PATH, parse_dates=['Date'])
stores_raw   = pd.read_csv(STORES_PATH)

df = (
    train_raw
    .merge(features_raw, on=['Store', 'Date', 'IsHoliday'], how='left')
    .merge(stores_raw,   on=['Store'],                       how='left')
)

wandb.log({
    'raw_rows'  : df.shape[0],
    'raw_cols'  : df.shape[1],
    'stores'    : df['Store'].nunique(),
    'depts'     : df['Dept'].nunique(),
    'date_min'  : str(df['Date'].min().date()),
    'date_max'  : str(df['Date'].max().date()),
})

# ── Null check ─────────────────────────────────────────────────
null_pct = df.isnull().mean().mul(100).round(2)
null_df  = null_pct[null_pct > 0].reset_index()
null_df.columns = ['column', 'null_pct']
wandb.log({'null_percentages': wandb.Table(dataframe=null_df)})
print('Nulls (%):')
print(null_df.to_string(index=False))

# ── Format for neuralforecast: unique_id | ds | y ──────────────
# N-BEATS only needs the raw sales series — no tabular features
df['unique_id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)
df_nf = (
    df[['unique_id', 'Date', 'Weekly_Sales', 'IsHoliday']]
    .rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)

# ── Drop series too short for the lookback window ──────────────
min_len    = INPUT_SIZE + H
series_len = df_nf.groupby('unique_id')['ds'].count()
valid_ids  = series_len[series_len >= min_len].index
dropped    = series_len[series_len < min_len].index.tolist()
df_nf      = df_nf[df_nf['unique_id'].isin(valid_ids)].reset_index(drop=True)

wandb.log({
    'total_series'    : len(series_len),
    'valid_series'    : len(valid_ids),
    'dropped_series'  : len(dropped),
    'dropped_ids'     : dropped,
    'min_series_len'  : int(series_len[valid_ids].min()),
    'max_series_len'  : int(series_len[valid_ids].max()),
})

print(f'\nValid series : {len(valid_ids)}')
print(f'Dropped      : {len(dropped)} (shorter than {min_len} weeks)')

# ── Train / val split ──────────────────────────────────────────
df_train = df_nf[df_nf['ds'] <  VAL_START].copy()
df_val   = df_nf[df_nf['ds'] >= VAL_START].copy()

top_stores = (
    df.groupby('Store')['Weekly_Sales']
    .sum()
    .nlargest(100)
    .index
)
df_train = df_train[df_train['unique_id'].str.split('_').str[0].astype(int).isin(top_stores)]
df_val   = df_val[df_val['unique_id'].str.split('_').str[0].astype(int).isin(top_stores)]

wandb.log({
    'train_rows'     : len(df_train),
    'val_rows'       : len(df_val),
    'train_date_min' : str(df_train['ds'].min().date()),
    'train_date_max' : str(df_train['ds'].max().date()),
    'val_date_min'   : str(df_val['ds'].min().date()),
    'val_date_max'   : str(df_val['ds'].max().date()),
    'val_start'      : VAL_START,
    'horizon_weeks'  : H,
    'lookback_weeks' : INPUT_SIZE,
})

print(f'Train : {df_train["ds"].min().date()} → {df_train["ds"].max().date()}  ({len(df_train):,} rows)')
print(f'Val   : {df_val["ds"].min().date()} → {df_val["ds"].max().date()}  ({len(df_val):,} rows)')

wandb.finish()

baseline_wmae,▁
best_val_mae,▁
best_val_wmae,▁
trials_completed,▁
wmae_improvement,▁
baseline_wmae,1360.78576
best_params,"{'input_size': 104, ..."
best_val_mae,1375.83594
best_val_wmae,1360.67424
trials_completed,15
wmae_improvement,0.11152


Nulls (%):
   column  null_pct
MarkDown1     64.26
MarkDown2     73.61
MarkDown3     67.48
MarkDown4     67.98
MarkDown5     64.08

Valid series : 2983
Dropped      : 348 (shorter than 56 weeks)
Train : 2010-02-05 → 2012-03-30  (328,806 rows)
Val   : 2012-04-06 → 2012-10-26  (87,383 rows)


depts,▁
dropped_series,▁
horizon_weeks,▁
lookback_weeks,▁
max_series_len,▁
min_series_len,▁
raw_cols,▁
raw_rows,▁
stores,▁
total_series,▁
+3,...


## 2. Training

In [22]:
# WMAE — the official Walmart competition metric
# Holiday weeks are weighted 5x more than regular weeks
def wmae(y_true: np.ndarray, y_pred: np.ndarray, is_holiday: np.ndarray) -> float:
    weights = np.where(is_holiday, 5.0, 1.0)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


# Full series (train + val) — cross_validation slices windows internally
df_full = pd.concat([df_train, df_val]).sort_values(['unique_id', 'ds']).reset_index(drop=True)

# Number of rolling windows to cover the validation period
# val period ≈ 28-30 weeks, step_size = H = 4 → 7 windows
N_WINDOWS = 7

def evaluate_cv(nf: NeuralForecast, model_col: str) -> tuple:
    """
    Rolling-window evaluation over the whole validation period.
    Trains once, then forecasts 7 consecutive 4-week windows.
    Returns (wmae, mae, eval_df).
    """
    cv_df = nf.cross_validation(
        df        = df_full[['unique_id', 'ds', 'y']],
        n_windows = N_WINDOWS,
        step_size = H,
    )
    cv_df = cv_df.reset_index() if 'unique_id' not in cv_df.columns else cv_df

    # attach IsHoliday back
    eval_df = cv_df.merge(
        df_full[['unique_id', 'ds', 'IsHoliday']],
        on=['unique_id', 'ds'], how='left'
    ).rename(columns={model_col: 'y_pred'})

    score_wmae = wmae(eval_df['y'].values, eval_df['y_pred'].values, eval_df['IsHoliday'].values)
    score_mae  = float(np.mean(np.abs(eval_df['y'].values - eval_df['y_pred'].values)))
    return score_wmae, score_mae, eval_df

In [23]:
# ── Baseline: single N-BEATS with sensible defaults ────────────
# Run this first to verify the pipeline works and get a reference WMAE
# before spending time on hyperparameter search
run_baseline = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'training',
    name     = 'NBEATS_Baseline',
    config   = {
        'input_size'    : INPUT_SIZE,
        'h'             : H,
        'stack_types'   : ['trend', 'seasonality'],
        'n_blocks'      : [3, 3],
        'mlp_units'     : [[512, 512], [512, 512]],
        'n_harmonics'   : 2,
        'n_polynomials' : 2,
        'max_steps'     : 200,
        'learning_rate' : 1e-3,
        'batch_size'    : 32,
        'loss'          : 'MAE',
        'variant'       : 'interpretable',
    }
)

baseline_model = NBEATS(
    h               = H,
    input_size      = INPUT_SIZE,
    stack_types     = ['trend', 'seasonality'],
    n_blocks        = [3, 3],
    mlp_units       = [[512, 512], [512, 512]],
    n_harmonics     = 2,
    n_polynomials   = 2,
    loss            = MAE(),
    max_steps       = 500,
    learning_rate   = 1e-3,
    batch_size      = 32,
    val_check_steps = 50,
    logger          = True,
)

nf_baseline = NeuralForecast(models=[baseline_model], freq='W-FRI')
baseline_wmae, baseline_mae, eval_baseline = evaluate_cv(nf_baseline, 'NBEATS')

wandb.log({
    'val_wmae' : baseline_wmae,
    'val_mae'  : baseline_mae,
})

print(f'Baseline WMAE : {baseline_wmae:,.2f}')
print(f'Baseline MAE  : {baseline_mae:,.2f}')

wandb.finish()

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  3.3 M │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 3.3 M                                                                                            
Non-trainable params: 1.5 K                                                                                        
Total params: 3.3 M                                                                                                
Total estimated model params size (MB): 13.376                                                                     
Modules in train mode: 52                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

Baseline WMAE : 1,360.79
Baseline MAE  : 1,373.70


val_mae,▁
val_wmae,▁
val_mae,1373.69651
val_wmae,1360.78576


In [ ]:
# ── Manual hyperparameter search (random search) ───────────────
# AutoNBEATS has a bug with n_blocks serialization in Ray Tune,
# so we implement random search manually — same idea as GridSearchCV
# but sampling random combinations instead of the full grid.
run_search = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'training',
    name     = 'NBEATS_HPSearch',
    config   = {
        'n_trials'    : N_TRIALS,
        'h'           : H,
        'search_type' : 'random',
        'search_space': {
            'input_size'    : [26, 52, 104],
            'mlp_units'     : [128, 256, 512],
            'n_harmonics'   : [1, 2, 4],
            'n_polynomials' : [1, 2, 3],
            'learning_rate' : [1e-4, 5e-4, 1e-3],
            'batch_size'    : [16, 32, 64],
        }
    }
)

from itertools import product
import random
random.seed(42)

search_space = {
    'input_size'    : [26, 52, 104],
    'mlp_units'     : [[128, 128], [256, 256], [512, 512]],
    'n_harmonics'   : [1, 2, 4],
    'n_polynomials' : [1, 2, 3],
    'learning_rate' : [1e-4, 5e-4, 1e-3],
    'batch_size'    : [16, 32, 64],
}

all_combos = list(product(*search_space.values()))
sampled    = random.sample(all_combos, min(N_TRIALS, len(all_combos)))
keys       = list(search_space.keys())

best_wmae   = float('inf')
best_mae    = None
best_params = None
nf_auto     = None
trial_results = []

for i, combo in enumerate(sampled):
    params = dict(zip(keys, combo))
    print(f'\n── Trial {i+1}/{N_TRIALS}: {params}')

    model = NBEATS(
        h               = H,
        input_size      = params['input_size'],
        stack_types     = ['trend', 'seasonality'],
        n_blocks        = [3, 3],
        mlp_units       = [params['mlp_units']] * 2,
        n_harmonics     = params['n_harmonics'],
        n_polynomials   = params['n_polynomials'],
        loss            = MAE(),
        max_steps       = 500,
        learning_rate   = params['learning_rate'],
        batch_size      = params['batch_size'],
        val_check_steps = 50,
        start_padding_enabled = True,
    )

    nf = NeuralForecast(models=[model], freq='W-FRI')
    trial_wmae, trial_mae, _ = evaluate_cv(nf, 'NBEATS')

    print(f'   WMAE: {trial_wmae:,.2f}   MAE: {trial_mae:,.2f}')
    trial_results.append({
        'trial'         : i + 1,
        'input_size'    : params['input_size'],
        'mlp_units'     : str(params['mlp_units']),
        'n_harmonics'   : params['n_harmonics'],
        'n_polynomials' : params['n_polynomials'],
        'learning_rate' : params['learning_rate'],
        'batch_size'    : params['batch_size'],
        'wmae'          : trial_wmae,
        'mae'           : trial_mae,
    })

    if trial_wmae < best_wmae:
        best_wmae   = trial_wmae
        best_mae    = trial_mae
        best_params = params
        nf_auto     = nf     # keep the best fitted model

trials_df = pd.DataFrame(trial_results)
_, _, eval_best = evaluate(nf_auto, 'NBEATS')

wandb.log({
    'best_val_wmae'    : best_wmae,
    'best_val_mae'     : best_mae,
    'baseline_wmae'    : baseline_wmae,
    'wmae_improvement' : baseline_wmae - best_wmae,
    'best_params'      : str(best_params),
    'trials_completed' : 15,
    'all_trials'       : wandb.Table(dataframe=trials_df),
})

print(f'\n════════════════════════════════════')
print(f'Best WMAE     : {best_wmae:,.2f}')
print(f'Baseline WMAE : {baseline_wmae:,.2f}')
print(f'Improvement   : {baseline_wmae - best_wmae:,.2f}')
print(f'Best params   : {best_params}')

# Prediction plots
sample_ids = eval_best['unique_id'].unique()[:6]
fig, axes  = plt.subplots(3, 2, figsize=(14, 10))
axes       = axes.flatten()

for ax, uid in zip(axes, sample_ids):
    history = df_train[df_train['unique_id'] == uid].tail(52)
    actual  = eval_best[eval_best['unique_id'] == uid]
    ax.plot(history['ds'], history['y'],     label='History', color='steelblue')
    ax.plot(actual['ds'],  actual['y'],      label='Actual',  color='black')
    ax.plot(actual['ds'],  actual['y_pred'], label='N-BEATS', color='tomato', linestyle='--')
    hol = actual[actual['IsHoliday'] == 1]
    ax.scatter(hol['ds'], hol['y'], color='gold', zorder=5, s=40, label='Holiday')
    ax.set_title(f'Store-Dept {uid}', fontsize=10)
    ax.legend(fontsize=7)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('N-BEATS Best Model — Validation Predictions', fontsize=13)
plt.tight_layout()
wandb.log({'prediction_plots': wandb.Image(fig)})
plt.show()

# Per-series WMAE
per_series = (
    eval_best
    .groupby('unique_id')
    .apply(lambda g: wmae(g['y'].values, g['y_pred'].values, g['IsHoliday'].values))
    .reset_index()
    .rename(columns={0: 'wmae'})
    .sort_values('wmae', ascending=False)
)
wandb.log({'per_series_wmae': wandb.Table(dataframe=per_series)})

wandb.finish()

## 3. Save Best Model to MLflow Registry

In [25]:
import dagshub
dagshub.init(repo_owner='ikvas22', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=3122e55a-7b4e-412a-9945-957144e6f01e&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=435f818095b940b876d756881b0d01e4cfb81ba6122a0cf73ca9326c0a7e7226




Accessing as ikvas22

Initialized MLflow to track repo "ikvas22/Walmart-Recruiting---Store-Sales-Forecasting"

Repository ikvas22/Walmart-Recruiting---Store-Sales-Forecasting initialized!

In [26]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)

2026/07/02 18:54:25 INFO mlflow.tracking.fluent: Experiment with name 'NBEATS_Training' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/077fbbafdaa849ab9b8feb3afcb96c52', creation_time=1783018465081, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1783018465081, lifecycle_stage='active', name='NBEATS_Training', tags={}, trace_location=None, workspace='default'>

In [27]:
# Wrapper so the registered model accepts raw unprocessed Walmart data
# and can be called directly in model_inference.ipynb without any preprocessing
class NBEATSWrapper(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        with open(context.artifacts['nf_model'], 'rb') as f:
            self.nf = pickle.load(f)

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        """
        Accepts raw DataFrame with [Store, Dept, Date, Weekly_Sales].
        Returns [Store, Dept, Date, Weekly_Sales_pred].
        """
        df_in              = model_input.copy()
        df_in['Date']      = pd.to_datetime(df_in['Date'])
        df_in['unique_id'] = df_in['Store'].astype(str) + '_' + df_in['Dept'].astype(str)
        df_nf_in = (
            df_in
            .rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
            [['unique_id', 'ds', 'y']]
            .sort_values(['unique_id', 'ds'])
        )
        preds    = self.nf.predict(df=df_nf_in)
        pred_col = [c for c in preds.columns if c not in ['unique_id', 'ds']][0]
        preds    = preds.rename(columns={pred_col: 'Weekly_Sales_pred'})
        preds[['Store', 'Dept']] = preds['unique_id'].str.split('_', expand=True).astype(int)
        return preds[['Store', 'Dept', 'ds', 'Weekly_Sales_pred']].rename(columns={'ds': 'Date'})


os.makedirs('mlflow_artifacts', exist_ok=True)
model_path = 'mlflow_artifacts/nf_auto_nbeats.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(nf_auto, f)

with mlflow.start_run(run_name='NBEATS_Best_Model') as run:
    mlflow.log_params(best_params)
    mlflow.log_metrics({
        'val_wmae'         : best_wmae,
        'val_mae'          : best_mae,
        'baseline_wmae'    : baseline_wmae,
        'wmae_improvement' : baseline_wmae - best_wmae,
        'n_trials'         : N_TRIALS,
    })
    mlflow.pyfunc.log_model(
        artifact_path         = 'nbeats_model',
        python_model          = NBEATSWrapper(),
        artifacts             = {'nf_model': model_path},
        registered_model_name = MLFLOW_MODEL_NAME,
    )
    print(f'Registered: {MLFLOW_MODEL_NAME}')
    print(f'Run ID    : {run.info.run_id}')
    print(f'Best WMAE : {best_wmae:,.2f}')

2026/07/02 18:54:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/02 18:54:30 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/07/02 18:54:30 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.core.frame.DataFrame'>, output: <class 'pandas.core.frame.DataFrame'>> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.


Successfully registered model 'nbeats_walmart_best'.
2026/07/02 18:54:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nbeats_walmart_best, version 1
Created version '1' of model 'nbeats_walmart_best'.


Registered: nbeats_walmart_best
Run ID    : 2a664ef8f7544a14b73fee0168ea84f1
Best WMAE : 1,360.67
🏃 View run NBEATS_Best_Model at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/0/runs/2a664ef8f7544a14b73fee0168ea84f1
🧪 View experiment at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/0


In [ ]:
# Sanity check: load from registry and predict on raw data
# This is exactly how model_inference.ipynb will use it
loaded = mlflow.pyfunc.load_model(f'models:/{MLFLOW_MODEL_NAME}/latest')
sample = train_raw[train_raw['Store'] == 1][['Store', 'Dept', 'Date', 'Weekly_Sales']].head(100)
result = loaded.predict(sample)
print('Registry load OK')
print(result.head())

In [29]:
wandb.finish()

baseline_wmae,▁
best_val_mae,▁
best_val_wmae,▁
trials_completed,▁
wmae_improvement,▁
baseline_wmae,1360.78576
best_params,"{'input_size': 104, ..."
best_val_mae,1375.83594
best_val_wmae,1360.67424
trials_completed,15
wmae_improvement,0.11152
